# 01. Setup & Baseline (Image × Regression Cell)

**Phase 1**: HF 데이터셋(UTKFace/SCUT-FBP5500) 로드 → DSC 회귀 베이스라인 → 회귀모델 5개 sanity.

DSC v5 framework — image × regression cell (ADR-018 사전등록). 이미지 분류(ADR-014) 노트북 미러.

In [ ]:
# ============================================================
# 0-1. Drive 마운트 + GPU 확인
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import numpy as np
import pandas as pd
import torch

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image_regression'
POLLUTED_DIR = f'{BASE}/data/image_regression_polluted'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch: {torch.__version__}')

In [ ]:
# ============================================================
# 0-2. 의존성 설치 (Colab)
# ============================================================
%pip install -q datasets timm imagehash opencv-python-headless

## 1. 데이터셋 로드 — UTKFace(age) / SCUT-FBP5500(beauty)

In [ ]:
# ============================================================
# 사전등록 메타 (ADR-018) — HuggingFace datasets
# ============================================================
DATASETS = {
    'UTKFace':       {'hf': 'Subh775/UTKFace_demographics_V1', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500':  {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
TUNE_DS, HELD_DS = 'UTKFace', 'SCUT_FBP5500'
SAMPLE_CAP = 5000   # DSC 계산용 (메모리/시간 절약)
RANDOM_SEED = 42
ML_SPLIT_SEED = 1
ML_TEST_SIZE = 0.2
print(f'데이터셋: {list(DATASETS.keys())} (튜닝={TUNE_DS}, held-out={HELD_DS})')

In [ ]:
# ============================================================
# HF 데이터셋 로더 + numpy 변환 (split-first: HF는 train split만 → 자체 분할)
# ============================================================
from datasets import load_dataset

def load_hf_split(ds_name):
    """HF 데이터셋 로드 후 train/test 인덱스 분할 (회귀 — stratify 없음)."""
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    from sklearn.model_selection import train_test_split
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE,
                                      random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx

def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    """주어진 인덱스(예: train) 중 sample_cap개를 (images[np.uint8], targets[float])로."""
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]
        img = ex[meta['image_col']]
        if hasattr(img, 'convert'):
            img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8))
        targets.append(float(ex[meta['target']]))
    return images, targets, indices

## 2. DSC 회귀 베이스라인 (clean train)

In [ ]:
# ============================================================
# 2-1. DSC framework import
# ============================================================
from dsc_framework import compute_dsc_image_regression, DEFAULT_WEIGHTS_IMAGE_REG
print('image regression DSC 엔진 import 완료')
print(f'사전등록(fallback) 가중치 (sum={sum(DEFAULT_WEIGHTS_IMAGE_REG.values()):.2f}):')
for k, v in DEFAULT_WEIGHTS_IMAGE_REG.items():
    print(f'  {k:<30s} {v:.2f}')

In [ ]:
# ============================================================
# 2-2. 데이터셋별 베이스라인 DSC (train clean)
# ============================================================
baseline_dsc_rows = []
for ds_name in DATASETS:
    print(f'\n{ds_name} 로드 + DSC...')
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    images, targets, _ = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    res = compute_dsc_image_regression(images, targets, sample_cap=SAMPLE_CAP)
    res.pop('metrics', None)
    print(f'  train={len(tr_idx)} test={len(te_idx)} sample={len(images)} → DSC={res["score"]} ({res["grade"]})')
    baseline_dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **res})

df_baseline_dsc = pd.DataFrame(baseline_dsc_rows)
df_baseline_dsc

## 3. 결과 저장 (이미지 회귀 cell — 별도 파일)

In [ ]:
# ============================================================
# 3-1. baseline DSC 저장
# ============================================================
dsc_path = f'{RESULTS_DIR}/dsc_scores_image_regression.csv'
def upsert_baseline(path, new_df):
    if os.path.isfile(path):
        ex = pd.read_csv(path)
        ex = ex[~((ex.polluter == 'none') & (ex.level == 0.0))]
        for c in new_df.columns:
            if c not in ex.columns: ex[c] = pd.NA
        combined = pd.concat([ex[new_df.columns], new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(path, index=False); return len(combined)
n = upsert_baseline(dsc_path, df_baseline_dsc)
print(f'DSC 저장: {dsc_path} (총 {n}건)')
print('--- 노트북 01 image regression 완료 → 02 실행 ---')